# 04 · Conformal Prediction: distribution-free intervals around any model

Conformal prediction turns **any point (or quantile) forecaster** into prediction intervals with a
**finite-sample coverage guarantee**. It makes no distributional assumption, only exchangeability.

## Theory

### Split (inductive) conformal
1. Fit a model $\hat f$ on a training set.
2. On a separate **calibration** set compute **non-conformity scores**, e.g. $s_i=|y_i-\hat f(x_i)|$.
3. Let $\hat q$ = the $\lceil (n+1)(1-\alpha)\rceil$-th smallest of the $n$ scores.
4. Interval for a new point: $\hat f(x)\pm\hat q$.

**Guarantee.** If calibration and test points are exchangeable:
$$1-\alpha\ \le\ \Pr\big(Y_{new}\in \hat C(X_{new})\big)\ \le\ 1-\alpha+\tfrac{1}{n+1}.$$
The "+1" in $n+1$ matters. Using the plain empirical $(1-\alpha)$ quantile under-covers for small $n$.
That was exactly the problem with the empirical-error intervals in nb 03.

### Variants
| Variant | Score | Effect |
|---|---|---|
| Absolute residual | $\lvert y-\hat y\rvert$ | symmetric, constant width |
| Signed / two one-sided | $y-\hat y$ at $\alpha/2$ each tail | asymmetric intervals |
| Normalised | $\lvert y-\hat y\rvert/\hat\sigma(x)$ | width adapts to predicted difficulty |
| **CQR** (Romano et al. 2019) | $\max(\hat q_{lo}-y,\ y-\hat q_{hi})$ | conformalises quantile regression: adaptive **and** guaranteed |

### Time series break exchangeability
Trends, seasonality, autocorrelation and regime changes violate exchangeability. Practical remedies:
* **Per-horizon** score sets: calibrate $h$-step errors separately for each $h$.
* **Rolling calibration window** (only the most recent $K$ errors), as in EnbPI (Xu & Xie 2021).
* **Adaptive Conformal Inference, ACI** (Gibbs & Candès 2021): update the miscoverage level online,
  $$\alpha_{t+1}=\alpha_t+\gamma\,(\alpha-\text{err}_t),\qquad \text{err}_t=\mathbb 1\{y_t\notin\hat C_t\}.$$
  The long-run coverage converges to $1-\alpha$ **for any sequence**, even adversarial ones.
  Conformal PID control (Angelopoulos et al. 2023) extends this idea.

Guarantees are **marginal** (on average over time / inputs), not conditional on a particular month.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.regression.quantile_regression import QuantReg
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(4)

def conformal_quantile(scores, alpha):
    """The ceil((n+1)(1-alpha))-th smallest score (inf if not enough scores)."""
    scores = np.sort(np.asarray(scores))
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else scores[k - 1]

## 1. The finite-sample correction, verified by simulation
Calibration scores are i.i.d. $|t_5|$. We compare the plain empirical 90% quantile with the conformal quantile.

In [2]:
alpha, trials = 0.1, 20_000
rows = []
for n_cal in [9, 19, 29, 49, 99, 499]:
    cal = np.abs(rng.standard_t(5, size=(trials, n_cal)))
    new = np.abs(rng.standard_t(5, size=trials))
    q_naive = np.quantile(cal, 1 - alpha, axis=1)
    q_conf = np.array([conformal_quantile(c, alpha) for c in cal])
    rows.append({"n_cal": n_cal, "naive coverage": np.mean(new <= q_naive), "conformal coverage": np.mean(new <= q_conf),
                 "theoretical upper bound": 1 - alpha + 1 / (n_cal + 1)})
pd.DataFrame(rows).set_index("n_cal")

,naive coverage,conformal coverage,theoretical upper bound
n_cal,,,
9,0.838,0.901,1.000
19,0.865,0.901,0.950
29,0.877,0.901,0.933
49,0.884,0.901,0.920
99,0.893,0.900,0.910
499,0.899,0.901,0.902


## 2. Conformal intervals around a monthly SARIMA (per-horizon, rolling calibration)

Rolling backtest on retail Total: at each origin we refit an airline SARIMA on log-sales and forecast 12 months ahead.
For each horizon $h$ we keep the **most recent $K=36$ out-of-sample errors** whose targets are already observed
(no look-ahead) and build:
* **Empirical quantiles** (plain `np.quantile`, the nb 03 method)
* **Conformal, absolute score**: symmetric $\pm\hat q$
* **Conformal, signed**: lower bound from $-e$ at $\alpha/2$, upper bound from $e$ at $\alpha/2$
* **Analytic SARIMA** for reference

In [3]:
y = pf.load_retail_total(); ly = np.log(y)
H, K, first, eval_from = 12, 36, 36, 72
err, fcs = {}, {}
for o in range(first, len(y)):
    hmax = min(H, len(y) - o)
    res = SARIMAX(ly.iloc[:o], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    fc = res.get_forecast(hmax)
    m, s = np.asarray(fc.predicted_mean), np.sqrt(np.asarray(fc.var_pred_mean))
    for h in range(1, hmax + 1):
        fcs[(o, h)] = (m[h - 1], s[h - 1], ly.iloc[o + h - 1])
        err[(o, h)] = ly.iloc[o + h - 1] - m[h - 1]

rows = []
for (o, h), (m, s, yt) in fcs.items():
    if o < eval_from:
        continue
    past = np.array([err[(j, h)] for j in range(o - h, first - 1, -1) if (j, h) in err][:K])  # most recent K
    for lev in [0.8, 0.9]:
        a = 1 - lev
        z = stats.norm.ppf(1 - a / 2)
        cands = {
            "Analytic SARIMA": (m - z * s, m + z * s),
            "Empirical quantiles": (m + np.quantile(past, a / 2), m + np.quantile(past, 1 - a / 2)),
            "Conformal |e|": (m - conformal_quantile(np.abs(past), a), m + conformal_quantile(np.abs(past), a)),
            "Conformal signed": (m - conformal_quantile(-past, a / 2), m + conformal_quantile(past, a / 2)),
        }
        for name, (lo, hi) in cands.items():
            rows.append(dict(origin=o, h=h, level=lev, method=name, n_cal=len(past),
                             y=np.exp(yt), lo=np.exp(lo), hi=np.exp(hi)))
bt = pd.DataFrame(rows)
bt["covered"] = (bt.y >= bt.lo) & (bt.y <= bt.hi)
bt["width"] = bt.hi - bt.lo
bt["interval_score"] = pf.interval_score(bt.y, bt.lo, bt.hi, 1 - bt.level)
bt.groupby(["level", "method"])[["covered", "width", "interval_score"]].mean().round(3)

covered    width  interval_score
level method                                               
0.8   Analytic SARIMA        0.869  483.181         558.855
      Conformal signed       0.847  471.070         575.603
      Conformal |e|          0.829  430.369         537.998
      Empirical quantiles    0.773  381.755         550.164
0.9   Analytic SARIMA        0.947  621.404         682.651
      Conformal signed       0.967  725.781         760.575
      Conformal |e|          0.949  615.085         660.942
      Empirical quantiles    0.876  497.239         671.752

In [4]:
METHODS = ["Analytic SARIMA", "Empirical quantiles", "Conformal |e|", "Conformal signed"]
COLOR = dict(zip(METHODS, pf.SERIES))
g = bt[bt.level == 0.9].groupby(["method", "h"]).covered.mean().unstack(0)[METHODS]
fig = pf.add_lines_by(go.Figure(), g, COLOR, fmt=".2f")
fig.add_hline(y=0.9, line=dict(color=pf.INK["primary"], dash="dash", width=1))
fig.update_layout(title="90% intervals: coverage by horizon (dashed = nominal)", height=400, hovermode="x unified",
                  yaxis_range=[0.5, 1.02], xaxis_title="h (months)", yaxis_title="empirical coverage")
fig.show()

The conformal intervals fix the empirical method's small-sample under-coverage. The signed version allows
asymmetric intervals. With $n$ calibration errors you cannot reach levels above $n/(n+1)$, so for a 90% level
you need at least 9 errors per horizon, and for 95% at least 19.

## 3. Adaptive Conformal Inference under a regime change
A synthetic monthly series whose noise level **jumps by 2.5×** in month 180 (think of a new competitor, COVID, …).
Base forecaster: seasonal naive. We compare three ways of choosing the interval half-width $\hat q_t$:

* **Fixed split**: calibrated once on months 12–59, never updated
* **Rolling window**: conformal quantile of the last 36 scores
* **ACI**: conformal quantile of all past scores at an *adaptive* level $\alpha_t$ ($\gamma=0.03$)

In [5]:
n = 300
t = np.arange(n)
idx = pd.date_range("2000-01-01", periods=n, freq="MS")
noise_sd = np.where(t < 180, 1.0, 2.5)
ys = 50 + 0.02 * t + 8 * np.sin(2 * np.pi * t / 12) + rng.normal(0, noise_sd)
pred = np.r_[np.full(12, np.nan), ys[:-12]]           # seasonal naive
score = np.abs(ys - pred)

alpha, gamma, start = 0.1, 0.03, 60
q_fixed = conformal_quantile(score[12:start], alpha)
alpha_t = alpha
out = {"Fixed split": [], "Rolling window (36)": [], "ACI": []}
alphas = []
for i in range(start, n):
    past = score[12:i]
    q_roll = conformal_quantile(past[-36:], alpha)
    q_aci = np.inf if alpha_t <= 0 else (0.0 if alpha_t >= 1 else conformal_quantile(past, alpha_t))
    for name, q in [("Fixed split", q_fixed), ("Rolling window (36)", q_roll), ("ACI", q_aci)]:
        out[name].append(score[i] <= q)
    alphas.append(alpha_t)
    alpha_t = alpha_t + gamma * (alpha - (score[i] > q_aci))   # ACI update

cov = pd.DataFrame(out, index=idx[start:]).astype(float)
fig = pf.subplots(2, 1, titles=["Rolling 24-month coverage (dashed = 90% target)", "ACI's adaptive miscoverage level α_t"],
                  shared_xaxes=True, height=640)
pf.add_lines_by(fig, cov.rolling(24).mean(), dict(zip(cov.columns, pf.SERIES)), row=1, col=1, fmt=".2f", markers=False)
fig.add_hline(y=1 - alpha, line=dict(color=pf.INK["primary"], dash="dash", width=1), row=1, col=1)
fig.add_trace(go.Scatter(x=idx[start:], y=alphas, mode="lines", name="ACI α_t", line=dict(color=pf.SERIES[2], width=2),
                         showlegend=False, hovertemplate="α_t = %{y:.3f}<extra></extra>"), row=2, col=1)
fig.add_hline(y=alpha, line=dict(color=pf.INK["primary"], dash="dash", width=1), row=2, col=1)
pf.add_vline(fig, idx[180], 0.4, 1.0, "noise level jumps 2.5×", color=pf.INK["primary"], dash="dot", row=1, col=1)
pf.add_vline(fig, idx[180], min(alphas), max(alphas), "noise level jumps 2.5×", color=pf.INK["primary"], dash="dot",
             row=2, col=1, showlegend=False)
fig.update_layout(hovermode="x unified")
fig.show()

pd.DataFrame({"before break": cov[cov.index < idx[180]].mean(), "after break": cov[cov.index >= idx[180]].mean(),
              "overall": cov.mean()}).round(3)

,before break,after break,overall
Fixed split,0.967,0.633,0.800
Rolling window (36),0.892,0.858,0.875
ACI,0.917,0.875,0.896


The fixed calibration collapses after the break. The rolling window recovers slowly. ACI drives $\alpha_t$ down
(wider intervals) until coverage is restored. **Larger $\gamma$ adapts faster but gives jumpier intervals.**

## 4. Conformalized Quantile Regression (CQR)
In nb 02, linear quantile regression (trend + month dummies) **under-covered** in the test period.
CQR keeps the adaptive *shape* of the QR band but corrects its *width* using a calibration set:
$$E_i=\max(\hat q_{lo}(x_i)-y_i,\ y_i-\hat q_{hi}(x_i)),\qquad \hat C(x)=[\hat q_{lo}(x)-\hat Q_E,\ \hat q_{hi}(x)+\hat Q_E].$$
If the band was too wide, $\hat Q_E<0$ and CQR **shrinks** it.

In [6]:
tr, cal, te = ly[:"2020-12-01"], ly["2021-01-01":"2022-12-01"], ly["2023-01-01":]
X = lambda s: pf.seasonal_design(s.index, "2015-01-01")
a = 0.1
qlo_m = QuantReg(tr, X(tr)).fit(q=a / 2, max_iter=5000)
qhi_m = QuantReg(tr, X(tr)).fit(q=1 - a / 2, max_iter=5000)
E = np.maximum(np.asarray(qlo_m.predict(X(cal))) - cal.to_numpy(), cal.to_numpy() - np.asarray(qhi_m.predict(X(cal))))
QE = conformal_quantile(E, a)
lo_raw, hi_raw = np.asarray(qlo_m.predict(X(te))), np.asarray(qhi_m.predict(X(te)))
lo_cqr, hi_cqr = lo_raw - QE, hi_raw + QE
yt = np.exp(te.to_numpy())
print(f"calibration n={len(E)}, Q_E = {QE:+.4f} (log scale)")
for name, lo, hi in [("QR raw", lo_raw, hi_raw), ("CQR", lo_cqr, hi_cqr)]:
    print(f"{name:7s} coverage={pf.coverage(yt, np.exp(lo), np.exp(hi)):.2f}  "
          f"width={np.mean(np.exp(hi) - np.exp(lo)):.0f}  IS={pf.interval_score(yt, np.exp(lo), np.exp(hi), a).mean():.0f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=y.index[-48:-24], y=y[-48:-24], mode="lines", name="history", line=dict(color=pf.HISTORY, width=1.5)))
pf.add_band(fig, te.index, np.exp(lo_cqr), np.exp(hi_cqr), "CQR 90% (conformalized)", pf.SERIES[0], 0.18)
pf.add_band(fig, te.index, np.exp(lo_raw), np.exp(hi_raw), "QR 90% (raw)", pf.SERIES[0], 0.45)
fig.add_trace(go.Scatter(x=te.index, y=yt, mode="markers+lines", name="actual", line=dict(color=pf.ACTUAL, width=1),
                         marker=dict(color=pf.ACTUAL, size=8, line=dict(color=pf.INK["surface"], width=2)),
                         customdata=np.exp(np.c_[lo_raw, hi_raw, lo_cqr, hi_cqr]),
                         hovertemplate="actual %{y:,.0f}<br>QR [%{customdata[0]:,.0f}, %{customdata[1]:,.0f}]"
                                       "<br>CQR [%{customdata[2]:,.0f}, %{customdata[3]:,.0f}]<extra></extra>"))
fig.update_layout(title="CQR shifts both quantile-regression bounds by the calibration-set correction Q_E",
                  hovermode="x unified", yaxis_title="sales")
fig.show()

calibration n=24, Q_E = +0.0884 (log scale)
QR raw  coverage=0.62  width=297  IS=881
CQR     coverage=1.00  width=674  IS=674


CQR repairs the under-coverage and improves the interval score substantially. Here it over-corrects (wide intervals,
100% on 24 test points), because the 2021–22 calibration years happened to deviate more from the QR band than
2023–24 did. The guarantee is *marginal* and assumes exchangeability, which a trending monthly series only
approximates. A rolling calibration window (§2) or ACI (§3) is the time-series-friendly upgrade.

## 5. Takeaways
* Conformal prediction is a **wrapper**: keep your favourite model and get calibrated intervals.
* Always use the $\lceil (n+1)(1-\alpha)\rceil$ rank, not `np.quantile`, when calibration sets are small.
* For time series: calibrate **per horizon**, use **recent** errors, and add **ACI** when regimes change.
* CQR combines the adaptivity of quantile regression with the coverage guarantee.
* Guarantees are marginal. Coverage for a specific month type (e.g. Decembers) is not guaranteed.

### Exercises
1. Repeat §3 with γ ∈ {0.005, 0.01, 0.1}. Plot interval width over time.
2. Implement a normalised score $|e|/\hat\sigma_h$ using the analytic SARIMA σ in §2.
3. Use a seasonal calibration set (only past Decembers) for December forecasts: is coverage per month better?